In [1]:
# ============================================================
# FINMARK PREDICTIVE MODEL
# Course: MO-IT134 | Milestone: 1 | Section: BSIT-S3101
# Author: Paran, Ralph Matthew
#
# Goal: Predict whether a FinMark customer is likely to
# make a purchase using three datasets:
#   - customers_data   : company info (profit, address)
#   - products_data    : FinMark software products and prices
#   - transactions_data: records of every transaction
#
# Steps:
#   1. Load and examine the datasets
#   2. Clean missing and inconsistent values
#   3. Merge the three datasets into one table
#   4. Create the target variable (purchased: yes or no)
#   5. Select features that reflect purchasing behaviour
#   6. Compare multiple models and select the best one
#   7. Evaluate the final model and interpret results
# ============================================================

# pandas helps me work with tables of data (like Excel but in Python)
import pandas as pd

# numpy helps me do math and handle missing values
import numpy as np

# machine learning models I will compare
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# tools to prepare data and evaluate models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully!')

All libraries loaded successfully!


In [2]:
# ============================================================
# STEP 1 — LOAD DATASETS
#
# I load the three FinMark CSV files from my local folder.
# os.chdir tells Python which folder to look in so I do not
# need to type the full file path every time.
# ============================================================

import os

# point Python to the folder where my CSV files are saved
os.chdir(r'C:\Users\Matti\Documents\School')

# load each dataset into a variable
customers_df    = pd.read_csv('customers_data.csv')
products_df     = pd.read_csv('products_data.csv')
transactions_df = pd.read_csv('transactions_data.csv')

print('Datasets loaded successfully!')
print(f'   customers_data:    {customers_df.shape[0]} rows, {customers_df.shape[1]} columns')
print(f'   products_data:     {products_df.shape[0]} rows, {products_df.shape[1]} columns')
print(f'   transactions_data: {transactions_df.shape[0]} rows, {transactions_df.shape[1]} columns')

Datasets loaded successfully!
   customers_data:    100 rows, 4 columns
   products_data:     20 rows, 3 columns
   transactions_data: 10000 rows, 8 columns


In [3]:
# ============================================================
# STEP 2 — EXAMINE THE DATASETS
#
# Before cleaning or modeling, I inspect each dataset to understand:
#   - what columns exist and what they represent
#   - what data types each column uses (numbers, text, dates)
#   - how many values are missing
#
# This step shapes every decision I make later — what to clean,
# how to merge, and which features to use.
# ============================================================

for name, df in [('customers_data', customers_df), ('products_data', products_df), ('transactions_data', transactions_df)]:
    print(f'\n--- {name} ---')
    print('Column data types:')
    print(df.dtypes)
    print('\nMissing values per column:')
    print(df.isnull().sum())


--- customers_data ---
Column data types:
Company_ID        float64
Company_Name          str
Company_Profit    float64
Address               str
dtype: object

Missing values per column:
Company_ID        10
Company_Name       0
Company_Profit    12
Address            0
dtype: int64

--- products_data ---
Column data types:
Product_ID       float64
Product_Name         str
Product_Price        str
dtype: object

Missing values per column:
Product_ID       2
Product_Name     0
Product_Price    0
dtype: int64

--- transactions_data ---
Column data types:
Unnamed: 0          float64
Transaction_ID      float64
Company_ID          float64
Product_ID          float64
Quantity            float64
Transaction_Date        str
Product_Price       float64
Total_Cost          float64
dtype: object

Missing values per column:
Unnamed: 0          1000
Transaction_ID      1000
Company_ID          1000
Product_ID          1000
Quantity            1000
Transaction_Date       0
Product_Price       100

In [4]:
# preview the first 5 rows of each dataset so I can see what the data looks like
print('--- customers_data (first 5 rows) ---')
display(customers_df.head())

print('--- products_data (first 5 rows) ---')
display(products_df.head())

print('--- transactions_data (first 5 rows) ---')
display(transactions_df.head())

--- customers_data (first 5 rows) ---


,Company_ID,Company_Name,Company_Profit,Address
0,1.0,Tech Enterprises 1,80701.0,"EDSA, Barangay 606, Pasig, Philippines"
1,2.0,Global Partners 2,80511.0,"Commonwealth Ave, Barangay 789, Taguig, Philip..."
2,3.0,Quantum Associates 3,110664.0,"Roxas Blvd, Barangay 505, Pasig, Philippines"
3,4.0,Prime Network 4,NaN,"Alabang-Zapote Rd, Barangay 202, Taguig, Phili..."
4,5.0,Elite Ventures 5,69427.0,"Ayala Avenue, Barangay 101, Makati, Philippines"


--- products_data (first 5 rows) ---


,Product_ID,Product_Name,Product_Price
0,1.0,FinPredictor Suite,"?140,000"
1,2.0,MarketMinder Analytics,"?168,000"
2,3.0,TrendWise Forecaster,"?100,800"
3,4.0,CustomerScope Insights,"?123,200"
4,5.0,SalesSync Optimizer,"?84,000"


--- transactions_data (first 5 rows) ---


,Unnamed: 0,Transaction_ID,Company_ID,Product_ID,Quantity,Transaction_Date,Product_Price,Total_Cost
0,0.0,1.0,88.0,6.0,NaN,2024/03/26,194379.147964,1075200.0
1,1.0,2.0,29.0,19.0,16.0,"July 09, 2024",97930.993380,1428000.0
2,2.0,NaN,28.0,18.0,6.0,04/13/2024,126095.547778,940800.0
3,3.0,4.0,85.0,12.0,12.0,09-06-2023,NaN,1008000.0
4,4.0,5.0,47.0,3.0,8.0,07/06/2021,99575.609634,705600.0


In [5]:
# ============================================================
# STEP 3 — CLEAN customers_data
#
# Issues found:
#   - 10 rows are missing Company_ID
#   - 12 rows are missing Company_Profit
#
# Fix:
#   - Drop rows with no Company_ID (I cannot identify them)
#   - Fill missing Company_Profit with the median profit
#     (I use median instead of mean because financial data
#     often has extreme values that make the mean misleading)
#   - Convert Company_ID from float (1.0) to integer (1)
#     so it matches the format in transactions_data for merging
# ============================================================

customers_clean = customers_df.copy()
customers_clean = customers_clean.dropna(subset=['Company_ID'])
customers_clean['Company_Profit'] = customers_clean['Company_Profit'].fillna(customers_clean['Company_Profit'].median())
customers_clean['Company_ID'] = customers_clean['Company_ID'].astype(int)

print(f'customers_data: {customers_df.isnull().sum().sum()} missing values → {customers_clean.isnull().sum().sum()}')

customers_data: 22 missing values → 0


In [6]:
# ============================================================
# STEP 3 (continued) — CLEAN products_data
#
# Issues found:
#   - 2 rows are missing Product_ID
#   - Product_Price contains a currency symbol (e.g. '?140,000')
#     which prevents Python from reading it as a number
#
# Fix:
#   - Drop rows with no Product_ID
#   - Strip all non-numeric characters from Product_Price
#     so Python can treat the values as numbers
#   - Fill any remaining missing prices with the median price
#   - Convert Product_ID to integer for merging
# ============================================================

products_clean = products_df.copy()
products_clean = products_clean.dropna(subset=['Product_ID'])

# remove anything from Product_Price that is not a digit or a dot
products_clean['Product_Price'] = (
    products_clean['Product_Price']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
    .replace('', np.nan)
)
products_clean['Product_Price'] = pd.to_numeric(products_clean['Product_Price'], errors='coerce')
products_clean['Product_Price'] = products_clean['Product_Price'].fillna(products_clean['Product_Price'].median())
products_clean['Product_ID'] = products_clean['Product_ID'].astype(int)

print(f'products_data: {products_df.isnull().sum().sum()} missing values → {products_clean.isnull().sum().sum()}')

products_data: 2 missing values → 0


In [7]:
# ============================================================
# STEP 3 (continued) — CLEAN transactions_data
#
# Issues found:
#   - An unnamed column was auto-generated by pandas (just row numbers)
#   - 1,000 rows are missing across all columns
#
# Fix:
#   - Remove the unnamed column — I do not need it
#   - Drop rows missing Transaction_ID, Company_ID, or Product_ID
#     because these are the linking keys — without them the row
#     cannot be identified or merged with the other tables
#   - Fill missing Quantity, Product_Price, and Total_Cost with medians
#   - Convert all ID columns to integers for consistent merging
# ============================================================

transactions_clean = transactions_df.copy()
transactions_clean = transactions_clean.drop(columns=['Unnamed: 0'], errors='ignore')
transactions_clean = transactions_clean.dropna(subset=['Transaction_ID', 'Company_ID', 'Product_ID'])

for col in ['Quantity', 'Product_Price', 'Total_Cost']:
    transactions_clean[col] = transactions_clean[col].fillna(transactions_clean[col].median())

for col in ['Transaction_ID', 'Company_ID', 'Product_ID']:
    transactions_clean[col] = transactions_clean[col].astype(int)

print(f'transactions_data: {transactions_df.isnull().sum().sum()} missing values → {transactions_clean.isnull().sum().sum()}')

transactions_data: 7000 missing values → 0


In [8]:
# ============================================================
# STEP 4 — MERGE THE THREE DATASETS
#
# The three datasets are related through shared ID columns:
#   - transactions_data and customers_data share Company_ID
#   - transactions_data and products_data share Product_ID
#
# I start from transactions_data because it is the central table —
# every row represents one purchase event. I then bring in
# customer and product details to enrich each transaction.
#
# I use a LEFT JOIN so that all transaction rows are kept.
# If a transaction has no matching customer or product record,
# those columns return NaN — which I fix in the next step.
# A left join ensures I do not lose any transaction records.
# ============================================================

# join transactions with customer data on Company_ID
# this adds Company_Profit and other customer info to each transaction
df = transactions_clean.merge(customers_clean, on='Company_ID', how='left')

# join the result with product data on Product_ID
# this adds Product_Name to each transaction
df = df.merge(products_clean[['Product_ID', 'Product_Name']], on='Product_ID', how='left')

print(f'Merged dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
display(df.head())

Merged dataset: 7277 rows, 11 columns
Columns: ['Transaction_ID', 'Company_ID', 'Product_ID', 'Quantity', 'Transaction_Date', 'Product_Price', 'Total_Cost', 'Company_Name', 'Company_Profit', 'Address', 'Product_Name']


,Transaction_ID,Company_ID,Product_ID,Quantity,Transaction_Date,Product_Price,Total_Cost,Company_Name,Company_Profit,Address,Product_Name
0,1,88,6,10.0,2024/03/26,194379.147964,1075200.0,Elite Consulting 88,75950.0,"EDSA, Barangay 456, Taguig, Philippines",RevenueVue Dashboard
1,2,29,19,16.0,"July 09, 2024",97930.993380,1428000.0,Sky Industries 29,61952.0,"Edsa, brgy. 606, makati, philippines!",EcoNomix Modeler
2,4,85,12,12.0,09-06-2023,130556.442432,1008000.0,Green Ventures 85,113470.0,"EDSA, Barangay 707, Cebu City, Philippines",BudgetMaster Pro
3,5,47,3,8.0,07/06/2021,99575.609634,705600.0,Green Industries 47,31130.0,"Taft Ave, Barangay 707, Mandaluyong, Philippines",TrendWise Forecaster
4,6,80,11,4.0,2021/07/12,160658.675350,627200.0,Green Partners 80,111227.0,"Commonwealth Ave, Barangay 202, Manila, Philip...",OptiFlow Automation


In [9]:
# ============================================================
# STEP 4 (continued) — FIX POST-MERGE MISSING VALUES
#
# After merging, some transactions could not find a matching
# Company_ID or Product_ID in the other tables. Those rows
# returned NaN for the joined columns.
# I fill these gaps so the model receives no missing input values.
# ============================================================

# fill remaining missing numbers with the column median
for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

# fill remaining missing text with 'Unknown'
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')

print(f'Missing values after post-merge cleanup: {df.isnull().sum().sum()}')
print('Dataset is fully clean and ready for modeling!')

Missing values after post-merge cleanup: 0
Dataset is fully clean and ready for modeling!


In [10]:
# ============================================================
# STEP 5 — CREATE THE TARGET VARIABLE
#
# My model needs a clear yes/no answer to learn from.
# I create a column called 'purchased':
#   1 = the transaction had a Quantity greater than 0 (a real purchase)
#   0 = Quantity was 0 (no purchase was made)
#
# This binary column becomes the label my model tries to predict.
# ============================================================

df['purchased'] = (df['Quantity'] > 0).astype(int)

print('Target variable distribution:')
print(df['purchased'].value_counts())
print(f'\nOverall purchase rate: {df["purchased"].mean()*100:.1f}%')

Target variable distribution:
purchased
1    7172
0     105
Name: count, dtype: int64

Overall purchase rate: 98.6%


In [11]:
# ============================================================
# STEP 6 — SELECT FEATURES
#
# I choose the columns most likely to help the model learn
# purchasing behaviour. Here is the rationale for each:
#
#   Company_Profit  — more profitable companies are more likely
#                     to invest in software tools
#   Product_Price   — price sensitivity may influence whether
#                     a purchase is made
#   Quantity        — directly reflects purchasing volume
#   Total_Cost      — captures overall spend, signals purchase intent
#   Product_Name    — different products may have different purchase rates
#
# X = the input data (what the model learns from)
# y = the answer column (what the model is trying to predict)
# ============================================================

feature_cols = [
    'Company_Profit',
    'Product_Price',
    'Quantity',
    'Total_Cost',
    'Product_Name',
]

# keep only columns that exist in my merged dataset
feature_cols = [col for col in feature_cols if col in df.columns]

print(f'Selected {len(feature_cols)} features: {feature_cols}')

X = df[feature_cols].copy()
y = df['purchased']

Selected 5 features: ['Company_Profit', 'Product_Price', 'Quantity', 'Total_Cost', 'Product_Name']


In [12]:
# ============================================================
# STEP 7 — ENCODE TEXT AND SCALE FEATURES
#
# Machine learning models only understand numbers, not text.
# LabelEncoder converts each unique text value into a number.
# Example: 'FinPredictor Suite' → 0, 'MarketMinder' → 1
#
# StandardScaler then puts all numeric columns on the same scale.
# Without this, columns with large numbers (like Total_Cost in
# the millions) would unfairly dominate over smaller columns
# (like Quantity which is just 0-21).
# ============================================================

# encode text columns into numbers
le = LabelEncoder()
text_columns = X.select_dtypes(include='object').columns.tolist()
for col in text_columns:
    X[col] = le.fit_transform(X[col].astype(str))

print(f'Encoded text columns: {text_columns}')
print(f'Missing values remaining: {X.isnull().sum().sum()}')

Encoded text columns: ['Product_Name']
Missing values remaining: 0


In [13]:
# split into training (80%) and test (20%) sets
# stratify=y ensures both splits have a similar ratio of 1s and 0s
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# scale features — fit on training data only
# I never fit on test data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} rows')
print(f'Test set:     {X_test.shape[0]} rows')
print('Data is scaled and ready for modeling!')

Training set: 5821 rows
Test set:     1456 rows
Data is scaled and ready for modeling!


In [14]:
# ============================================================
# STEP 8 — COMPARE MULTIPLE MODELS
#
# Before choosing one algorithm, I test four common models:
#
#   Logistic Regression  — simple, interpretable, designed for
#                          binary yes/no classification problems
#   Decision Tree        — easy to visualize, shows clear rules
#   Random Forest        — usually high accuracy, handles complex
#                          patterns by combining many trees
#   K-Nearest Neighbors  — predicts based on similarity to nearby
#                          data points, good as a baseline
#
# I evaluate each on Accuracy and ROC-AUC Score.
# ROC-AUC measures how well the model separates buyers from
# non-buyers — a score of 1.0 is perfect, 0.5 is random guessing.
# ============================================================

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=100),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5)
}

results = []

for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred      = model.predict(X_test_scaled)
    y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
    acc     = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_prob)
    results.append({'Model': model_name, 'Accuracy': f'{acc*100:.2f}%', 'ROC-AUC': f'{roc_auc:.4f}'})
    print(f'{model_name}: Accuracy = {acc*100:.2f}%  |  ROC-AUC = {roc_auc:.4f}')

print('\nFull Comparison Table:')
display(pd.DataFrame(results))

Logistic Regression: Accuracy = 99.59%  |  ROC-AUC = 1.0000
Decision Tree: Accuracy = 100.00%  |  ROC-AUC = 1.0000
Random Forest: Accuracy = 100.00%  |  ROC-AUC = 1.0000
K-Nearest Neighbors: Accuracy = 98.56%  |  ROC-AUC = 0.8746

Full Comparison Table:


,Model,Accuracy,ROC-AUC
0,Logistic Regression,99.59%,1.0000
1,Decision Tree,100.00%,1.0000
2,Random Forest,100.00%,1.0000
3,K-Nearest Neighbors,98.56%,0.8746


In [15]:
# ============================================================
# STEP 9 — WHY I CHOOSE LOGISTIC REGRESSION
#
# After comparing all four models, I select Logistic Regression
# as my final model for the following reasons:
#
# 1. INTERPRETABILITY
#    Logistic Regression produces a coefficient (score) for each
#    feature that directly shows how much it increases or decreases
#    the likelihood of a purchase. This makes results easy to
#    explain to business stakeholders at FinMark.
#    Other models like Random Forest are 'black boxes' — they
#    predict well but cannot easily explain why.
#
# 2. DESIGNED FOR BINARY OUTCOMES
#    My question is simply: did the customer buy or not?
#    Logistic Regression is specifically built for yes/no problems.
#    It outputs a probability between 0 and 1 — so I can say
#    'this customer has a 78% chance of purchasing.'
#
# 3. AVOIDS OVERFITTING
#    Decision Trees tend to memorize the training data too well
#    and then struggle with new data (called overfitting).
#    Logistic Regression is simpler and generalizes better,
#    meaning it performs more reliably on unseen customers.
#
# 4. BUSINESS VALUE
#    FinMark needs to understand WHY customers buy, not just IF.
#    A model that says 'high-spending companies are more likely
#    to buy' is far more useful than one that silently predicts
#    yes or no without explanation.
#
# 5. COMPETITIVE PERFORMANCE
#    While other models may score slightly higher in accuracy,
#    the difference is typically small. Logistic Regression
#    delivers most of the accuracy with far more transparency —
#    a strong trade-off in a business setting.
# ============================================================

print('Final model selected: Logistic Regression')
print('Reason: Best balance of accuracy, interpretability, and business relevance.')

Final model selected: Logistic Regression
Reason: Best balance of accuracy, interpretability, and business relevance.


In [16]:
# ============================================================
# STEP 10 — TRAIN THE FINAL MODEL AND EVALUATE
#
# I train the final Logistic Regression model and evaluate it on
# the test set — data the model has never seen before.
#
# Metrics explained:
#   Accuracy      — overall % of correct predictions
#   ROC-AUC       — how well the model separates buyers from non-buyers
#                   (1.0 = perfect, 0.5 = random guessing)
#   Precision     — of all predicted purchases, how many were real?
#   Recall        — of all real purchases, how many did I catch?
#   Confusion Matrix — shows exactly where the model was right or wrong
# ============================================================

final_model = LogisticRegression(random_state=42, max_iter=1000)
final_model.fit(X_train_scaled, y_train)

y_pred      = final_model.predict(X_test_scaled)
y_pred_prob = final_model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc  = roc_auc_score(y_test, y_pred_prob)

print('=' * 50)
print('       FINAL MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'  Accuracy:      {accuracy*100:.2f}%')
print(f'  ROC-AUC Score: {roc_auc:.4f}  (closer to 1.0 = better)')
print('=' * 50)
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Purchased', 'Purchased']))
print('Confusion Matrix (rows = actual, columns = predicted):')
print(confusion_matrix(y_test, y_pred))

       FINAL MODEL EVALUATION RESULTS
  Accuracy:      99.59%
  ROC-AUC Score: 1.0000  (closer to 1.0 = better)

Detailed Classification Report:
               precision    recall  f1-score   support

Not Purchased       1.00      0.71      0.83        21
    Purchased       1.00      1.00      1.00      1435

     accuracy                           1.00      1456
    macro avg       1.00      0.86      0.92      1456
 weighted avg       1.00      1.00      1.00      1456

Confusion Matrix (rows = actual, columns = predicted):
[[  15    6]
 [   0 1435]]


In [17]:
# ============================================================
# STEP 11 — FEATURE IMPORTANCE
#
# Logistic Regression assigns a coefficient to each feature.
# A positive coefficient means the feature increases the
# likelihood of a purchase.
# A negative coefficient means it decreases it.
#
# This gives FinMark actionable insight — for example:
#   'High Total_Cost and Company_Profit increase purchase likelihood'
#   which tells the sales team to focus on high-spending companies.
# ============================================================

importance_table = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': final_model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print('Feature coefficients:')
print('(positive = increases purchase likelihood, negative = decreases it)')
print(importance_table.to_string(index=False))

Feature coefficients:
(positive = increases purchase likelihood, negative = decreases it)
       Feature  Coefficient
      Quantity     9.304263
    Total_Cost     0.683281
Company_Profit     0.195926
 Product_Price     0.085573
  Product_Name     0.074707


In [18]:
# ============================================================
# STEP 12 — SAVE ALL OUTPUT FILES
#
# I save the model predictions and all cleaned datasets
# so they can be submitted and reviewed on GitHub.
#
# finmark_predictions.csv     — model results with actual vs predicted
# customers_data_clean.csv    — cleaned version of customers_data
# products_data_clean.csv     — cleaned version of products_data
# transactions_data_clean.csv — cleaned version of transactions_data
# ============================================================

# save predictions
results_df = X_test.copy()
results_df['actual']               = y_test.values   # what actually happened
results_df['predicted']            = y_pred           # what the model predicted
results_df['purchase_probability'] = y_pred_prob      # model confidence score
results_df.to_csv('finmark_predictions.csv', index=False)

# save cleaned datasets
customers_clean.to_csv('customers_data_clean.csv', index=False)
products_clean.to_csv('products_data_clean.csv', index=False)
transactions_clean.to_csv('transactions_data_clean.csv', index=False)

print('All files saved to C:\\Users\\Matti\\Documents\\School')
print('')
print('Files saved:')
print('   finmark_predictions.csv      <- model predictions')
print('   customers_data_clean.csv     <- cleaned customers')
print('   products_data_clean.csv      <- cleaned products')
print('   transactions_data_clean.csv  <- cleaned transactions')

All files saved to C:\Users\Matti\Documents\School

Files saved:
   finmark_predictions.csv      <- model predictions
   customers_data_clean.csv     <- cleaned customers
   products_data_clean.csv      <- cleaned products
   transactions_data_clean.csv  <- cleaned transactions
